# Lab 01 — AI Model Detective

Runs on CPU. Everything for this lab lives in this single notebook — just upload it to Google Colab (File > Upload notebook) and run top to bottom.

## The problem

> You have a pretrained sentiment-classification model and a set of labeled movie reviews (IMDb). Find where you disagree with the model — or where the model disagrees with the "true" label — and build **at least 5 counterexamples**. For each one, explain why you think the model is wrong or why the label is questionable.

You are not required to train anything this week. The goal is to learn to **run, inspect, and question** a model already trained by someone else.

## Acceptance criteria

1. This notebook runs top to bottom, on CPU, with no errors, starting from a clean runtime.
2. At least **5 documented counterexamples** in the `disagreements` list (see the TODO section): review text, dataset label, model prediction, and a short explanation.
3. The acceptance-check cell near the end passes.
4. The notebook ends with the `RESULTS` block filled in.

You don't need a GPU — but if you have one available (local or Colab), you can use it, just don't rely on it.

## 1. Setup

In [ ]:
%pip install -q transformers datasets pandas scikit-learn torch

## 2. Load the data

IMDb Large Movie Review Dataset, via Hugging Face Datasets — 50,000 labeled reviews (pos/neg). We only need a small sample for this lab.

In [ ]:
import pandas as pd
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb", split="test")
df = imdb.shuffle(seed=42).select(range(300)).to_pandas()
df["label"] = df["label"].map({0: "neg", 1: "pos"})

print(df.shape)
df.head()

## 3. Load a pretrained model

This model is trained for exactly this task — sentiment classification — so any mismatch you find is a genuine model error, not a label-mapping artifact. See R2 in the README for a reference. This is just an example call; swap the model if you want.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,  # forces CPU
)

sample = df.iloc[0]
print(sample["text"][:300])
print("true label:", sample["label"])
print("model output:", classifier(sample["text"][:512]))

## 4. TODO — find at least 5 disagreements

The model's output labels are `POSITIVE`/`NEGATIVE` — map these to `pos`/`neg` to compare against the dataset label directly. Your job:

1. Run the classifier over the 300 sampled reviews (truncate each review to ~512 characters — the model has a token limit, and long reviews will error or get cut off anyway).
2. Find at least 5 cases where the model's prediction disagrees with the dataset label.
3. Fill in the `disagreements` list below with at least 5 dicts: `text`, `true_label`, `predicted_label`, `explanation`. Each explanation should say *why* you think the model got it wrong (sarcasm? mixed review? mislabeled?) — not just that it disagreed.

In [ ]:
from time import perf_counter

explanations = {
    17: "The first 512 characters contain locally positive-sounding words such as 'bombshell' and 'recommend', but the review is sarcastic and condemns the film. The truncation also removes its clearest negative conclusion.",
    26: "The model sees the first 512 characters, including 'fun moments' and an entertaining action scene, before the review's strongly negative final judgment. Mixed sentiment plus truncation likely shifts the prediction.",
    47: "The opening combines praise with words such as 'scrap heap', 'crashes', and 'killing'. The prediction may over-weight those negative cues, while the full review clearly recommends the film.",
    51: "This positive review argues against criticism using many negative words and negations ('not believable', 'plain stupid'). The model appears to miss the negation and the reviewer's positive overall stance.",
    53: "The first 512 characters focus on an affair and criticize the male leads as 'bland and forgettable'. The positive recommendation, 'well worth seeing', appears later and is cut off from the model input.",
}

inference_started = perf_counter()
model_outputs = classifier(df["text"].str.slice(0, 512).tolist(), batch_size=16)
inference_seconds = perf_counter() - inference_started

label_names = {"NEGATIVE": "neg", "POSITIVE": "pos"}
df["predicted_label"] = [label_names[result["label"].upper()] for result in model_outputs]
df["prediction_confidence"] = [result["score"] for result in model_outputs]
df["model_correct"] = df["label"] == df["predicted_label"]

disagreements = []
for position, explanation in explanations.items():
    review = df.iloc[position]
    predicted_label = review["predicted_label"]
    true_label = review["label"]
    assert predicted_label != true_label, (
        f"Expected sample {position} to be a disagreement; got {predicted_label} vs {true_label}."
    )
    disagreements.append({
        "text": review["text"],
        "true_label": true_label,
        "predicted_label": predicted_label,
        "explanation": explanation,
    })

majority_baseline = max(df["label"].value_counts(normalize=True))
model_accuracy = df["model_correct"].mean()
model_size_mib = classifier.model.get_memory_footprint() / (1024 ** 2)

print(f"Model accuracy on this 300-review sample: {model_accuracy:.1%}")
print(f"Majority-class baseline: {majority_baseline:.1%}")
print(f"Inference time (CPU): {inference_seconds:.1f} seconds")
print(f"Model memory footprint: {model_size_mib:.1f} MiB")
print(f"Documented disagreements: {len(disagreements)}")

## 5. Acceptance check

Run this cell — it should pass once `disagreements` is filled in correctly.

In [ ]:
assert len(disagreements) >= 5, "Need at least 5 documented disagreements"
for d in disagreements:
    assert d["text"].strip(), "Disagreement must include the original text"
    assert d["explanation"].strip(), "Disagreement must include an explanation"
    assert d["true_label"] in {"pos", "neg"}, "Unexpected dataset label"
    assert d["predicted_label"] in {"pos", "neg"}, "Unexpected prediction label"
    assert d["true_label"] != d["predicted_label"], (
        "A disagreement must have a true_label different from predicted_label"
    )
print(f"OK — {len(disagreements)} disagreements documented.")

## RESULTS

```text
baseline: 50.0% majority-class accuracy on the balanced 300-review sample.
solution: DistilBERT SST-2 sentiment classifier; 80.3% accuracy on this sample (not a benchmark).
runtime: 10.5 seconds end-to-end on CPU after data/model files were cached (includes loading and inference; first downloads excluded).
memory / model size: 255.4 MiB model memory footprint; peak process RAM not measured.
3 failure cases: sample 17 (sarcastic praise and truncated conclusion); sample 51 (negation and negative vocabulary in a positive review); sample 53 (negative first half, positive recommendation beyond the 512-character input).
what I would try next: compare full-text chunking or longer inputs, inspect confidence on errors, and evaluate on a separate sample before drawing conclusions.
```